# 3D Vertex Reconstruction in the TMS Detector using Event-Level Direct ML Regression

This notebook demonstrates the training and evaluation of the **Event-Level Direct Machine Learning Regression Model** for primary neutrino interaction vertex reconstruction in the **The Muon Spectrometer (TMS)**.

### Physics Geometry & Coordinate Transformation:
Scintillator strips in alternating planes are tilted at an angle $\theta = 3^\circ$ relative to the X-axis.
- $u = x \cos(\theta/2) + y \sin(\theta/2)$
- $v = x \cos(\theta/2) - y \sin(\theta/2)$

Given U-view and V-view fitted Hough lines:
- $u(z) = a_u z + b_u$
- $v(z) = a_v z + b_v$

The 3D coordinates are:
$$x(z) = \frac{a_u + a_v}{2 \cos(1.5^\circ)} z + \frac{b_u + b_v}{2 \cos(1.5^\circ)}$$
$$y(z) = \frac{a_u - a_v}{2 \sin(1.5^\circ)} z + \frac{b_u - b_v}{2 \sin(1.5^\circ)}$$

### Methodology:
Instead of filtering vertices using candidate-level classification, we compile the entire event's Hough line parameters into a global event-level feature vector (23 features). We then train three separate Gradient Boosting Regressors (`GradientBoostingRegressor`) to directly predict the true primary neutrino interaction vertex coordinates $(x_{true}, y_{true}, z_{true})$.

In [ ]:
import uproot
import awkward as ak
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
import joblib

plt.rcParams.update({
    "font.size": 11,
    "figure.figsize": (8, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 100,
})
print("Imports complete.")

In [ ]:
# 1. Load data from ROOT file
f = uproot.open("../Cut2.root")
t_lc = f["Line_Candidates;2"]
t_tr = f["Truth_Info;2"]
n_events = min(1000, t_lc.num_entries)

lc_arr = t_lc.arrays([
    "nLinesU", "nLinesV", 
    "SlopeU", "InterceptU", "SlopeV", "InterceptV",
    "FirstHoughHitU", "LastHoughHitU", "FirstHoughHitV", "LastHoughHitV",
    "nHitsInTrackU", "nHitsInTrackV",
    "TrackLengthU", "TrackLengthV",
    "TotalTrackEnergyU", "TotalTrackEnergyV"
], entry_stop=n_events)
tr_arr = t_tr.arrays(["NeutrinoX4"], entry_stop=n_events)
print(f"Loaded {n_events} events from ROOT file.")

In [ ]:
# 2. Reconstruct geometrical vertex and compile event-level features
theta = 3.0 * np.pi / 180.0
cos_half = np.cos(theta / 2.0)
sin_half = np.sin(theta / 2.0)

def get_line_start_end(event_id, view, idx):
    if view == "U":
        z1, u1 = lc_arr["FirstHoughHitU"][event_id][idx]
        z2, u2 = lc_arr["LastHoughHitU"][event_id][idx]
    else:
        z1, v1 = lc_arr["FirstHoughHitV"][event_id][idx]
        z2, v2 = lc_arr["LastHoughHitV"][event_id][idx]
    return min(z1, z2), max(z1, z2)

def fit_3d_line_from_2d(s_u, int_u, s_v, int_v):
    slope_x = (s_u + s_v) / (2.0 * cos_half)
    int_x = (int_u + int_v) / (2.0 * cos_half)
    slope_y = (s_u - s_v) / (2.0 * sin_half)
    int_y = (int_u - int_v) / (2.0 * sin_half)
    p0 = np.array([int_x, int_y, 0.0])
    d = np.array([slope_x, slope_y, 1.0])
    d = d / np.linalg.norm(d)
    return p0, d

def reconstruct_vertex_3d(lines):
    if len(lines) == 1:
        return lines[0][0]
    A = np.zeros((3, 3))
    b = np.zeros(3)
    for p, d in lines:
        proj = np.eye(3) - np.outer(d, d)
        A += proj
        b += proj @ p
    try:
        return np.linalg.solve(A, b)
    except:
        return np.mean([p for p, d in lines], axis=0)

dataset = []
for event_id in range(n_events):
    n_u = lc_arr["nLinesU"][event_id]
    n_v = lc_arr["nLinesV"][event_id]
    nx4 = tr_arr["NeutrinoX4"][event_id].tolist()
    true_x, true_y, true_z = nx4[0]*1000.0, nx4[1]*1000.0, nx4[2]*1000.0
    
    # Filter for TMS fiducial volume
    if not (11124.0 <= true_z <= 18544.0): 
        continue
    
    slopes_u = lc_arr["SlopeU"][event_id].tolist()
    intercepts_u = lc_arr["InterceptU"][event_id].tolist()
    slopes_v = lc_arr["SlopeV"][event_id].tolist()
    intercepts_v = lc_arr["InterceptV"][event_id].tolist()
    len_u = lc_arr["TrackLengthU"][event_id].tolist()
    len_v = lc_arr["TrackLengthV"][event_id].tolist()
    energy_u = lc_arr["TotalTrackEnergyU"][event_id].tolist()
    energy_v = lc_arr["TotalTrackEnergyV"][event_id].tolist()
    
    z_starts_u = [get_line_start_end(event_id, "U", i)[0] for i in range(n_u)]
    z_ends_u = [get_line_start_end(event_id, "U", i)[1] for i in range(n_u)]
    z_starts_v = [get_line_start_end(event_id, "V", j)[0] for j in range(n_v)]
    z_ends_v = [get_line_start_end(event_id, "V", j)[1] for j in range(n_v)]
    
    x_reco, y_reco, z_reco = 0.0, 0.0, 0.0
    matched_tracks = []
    if n_u > 0 and n_v > 0:
        u_used, v_used = set(), set()
        candidates = []
        for i in range(n_u):
            for j in range(n_v):
                z_start_diff = abs(z_starts_u[i] - z_starts_v[j])
                z_end_diff = abs(z_ends_u[i] - z_ends_v[j])
                if z_start_diff < 400.0 and z_end_diff < 600.0:
                    candidates.append((z_start_diff, i, j))
        candidates.sort()
        for diff, i, j in candidates:
            if i not in u_used and j not in v_used:
                u_used.add(i)
                v_used.add(j)
                p0, d = fit_3d_line_from_2d(slopes_u[i], intercepts_u[i], slopes_v[j], intercepts_v[j])
                z_start = (z_starts_u[i] + z_starts_v[j]) / 2.0
                x_start = ((slopes_u[i] + slopes_v[j]) / (2.0 * cos_half)) * z_start + ((intercepts_u[i] + intercepts_v[j]) / (2.0 * cos_half))
                y_start = ((slopes_u[i] - slopes_v[j]) / (2.0 * sin_half)) * z_start + ((intercepts_u[i] - intercepts_v[j]) / (2.0 * sin_half))
                start_pos = np.array([x_start, y_start, z_start])
                matched_tracks.append((start_pos, d))
        if matched_tracks:
            vtx = matched_tracks[0][0] if len(matched_tracks) == 1 else reconstruct_vertex_3d(matched_tracks)
            x_reco, y_reco, z_reco = vtx[0], vtx[1], vtx[2]
            
    if z_reco == 0.0:
        all_starts = z_starts_u + z_starts_v
        z_reco = np.mean(all_starts) if all_starts else 11500.0
        
    feat = {
        "x_reco": x_reco, "y_reco": y_reco, "z_reco": z_reco,
        "nLinesU": n_u, "nLinesV": n_v,
        "mean_len_u": np.mean(len_u) if len_u else 0.0,
        "max_len_u": np.max(len_u) if len_u else 0.0,
        "sum_len_u": np.sum(len_u) if len_u else 0.0,
        "mean_len_v": np.mean(len_v) if len_v else 0.0,
        "max_len_v": np.max(len_v) if len_v else 0.0,
        "sum_len_v": np.sum(len_v) if len_v else 0.0,
        "mean_energy_u": np.mean(energy_u) if energy_u else 0.0,
        "sum_energy_u": np.sum(energy_u) if energy_u else 0.0,
        "mean_energy_v": np.mean(energy_v) if energy_v else 0.0,
        "sum_energy_v": np.sum(energy_v) if energy_v else 0.0,
        "min_z_start_u": np.min(z_starts_u) if z_starts_u else 11500.0,
        "max_z_start_u": np.max(z_starts_u) if z_starts_u else 18500.0,
        "min_z_start_v": np.min(z_starts_v) if z_starts_v else 11500.0,
        "max_z_start_v": np.max(z_starts_v) if z_starts_v else 18500.0,
        "mean_slope_u": np.mean(slopes_u) if slopes_u else 0.0,
        "mean_slope_v": np.mean(slopes_v) if slopes_v else 0.0,
        "true_x": true_x, "true_y": true_y, "true_z": true_z
    }
    dataset.append(feat)

df = pd.DataFrame(dataset)
print(f"Compiled dataset for {len(df)} events inside TMS.")

In [ ]:
# 3. Train Gradient Boosting Regressors
feature_cols = [col for col in df.columns if not col.startswith("true_")]
X = df[feature_cols]
y_x = df["true_x"]
y_y = df["true_y"]
y_z = df["true_z"]

X_train, X_test, indices_train, indices_test = train_test_split(X, df.index, test_size=0.2, random_state=42)

reg_x = GradientBoostingRegressor(n_estimators=60, max_depth=3, learning_rate=0.08, random_state=42)
reg_y = GradientBoostingRegressor(n_estimators=60, max_depth=3, learning_rate=0.08, random_state=42)
reg_z = GradientBoostingRegressor(n_estimators=60, max_depth=3, learning_rate=0.08, random_state=42)

reg_x.fit(X_train, y_x.loc[indices_train])
reg_y.fit(X_train, y_y.loc[indices_train])
reg_z.fit(X_train, y_z.loc[indices_train])

print("Model training complete. Regressors fitted successfully.")

In [ ]:
# 4. Evaluate predictions on test set
df_test = df.loc[indices_test].copy()
X_test_feats = X.loc[indices_test]

pred_x = reg_x.predict(X_test_feats)
pred_y = reg_y.predict(X_test_feats)
pred_z = reg_z.predict(X_test_feats)

dx_before = df_test["x_reco"] - df_test["true_x"]
dy_before = df_test["y_reco"] - df_test["true_y"]
dz_before = df_test["z_reco"] - df_test["true_z"]
dr_before = np.sqrt(dx_before**2 + dy_before**2 + dz_before**2)

dx_after = pred_x - df_test["true_x"]
dy_after = pred_y - df_test["true_y"]
dz_after = pred_z - df_test["true_z"]
dr_after = np.sqrt(dx_after**2 + dy_after**2 + dz_after**2)

# Filter core 95% for statistical evaluation
thresh_before = np.percentile(dr_before, 95.0)
core_mask = dr_before <= thresh_before

print(f"Test set size: {len(df_test)}")
print("\n--- Residuals BEFORE Direct ML Regression ---")
print(pd.DataFrame({"dx": dx_before[core_mask], "dy": dy_before[core_mask], "dz": dz_before[core_mask], "dr": dr_before[core_mask]}).describe())

print("\n--- Residuals AFTER Direct ML Regression ---")
print(pd.DataFrame({"dx": dx_after[core_mask], "dy": dy_after[core_mask], "dz": dz_after[core_mask], "dr": dr_after[core_mask]}).describe())

In [ ]:
# 5. Plot Residuals Before vs. After
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(dy_before[core_mask], bins=20, color='red', alpha=0.5, edgecolor='black', label=f"Before: $\sigma$={np.std(dy_before[core_mask]):.1f} mm")
axes[0].hist(dy_after[core_mask], bins=20, color='green', alpha=0.5, edgecolor='black', label=f"After: $\sigma$={np.std(dy_after[core_mask]):.1f} mm")
axes[0].set_title("Y Residual (dy) Distribution")
axes[0].set_xlabel("dy (mm)")
axes[0].set_ylabel("Counts")
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].hist(dr_before[core_mask], bins=20, color='red', alpha=0.5, edgecolor='black', label=f"Before: median={np.median(dr_before[core_mask]):.1f} mm")
axes[1].hist(dr_after[core_mask], bins=20, color='green', alpha=0.5, edgecolor='black', label=f"After: median={np.median(dr_after[core_mask]):.1f} mm")
axes[1].set_title("3D Distance Error (dr) Distribution")
axes[1].set_xlabel("dr (mm)")
axes[1].set_ylabel("Counts")
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.suptitle("Event-Level Direct ML Regression Performance (Using All Info from U & V)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Plot Predicted vs. Actual Correlations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (coord, true_vals, pred_vals, color) in enumerate([
    ("X", df_test["true_x"], pred_x, "royalblue"),
    ("Y", df_test["true_y"], pred_y, "seagreen"),
    ("Z", df_test["true_z"], pred_z, "crimson")
]):
    axes[idx].scatter(true_vals, pred_vals, color=color, alpha=0.6, edgecolors='k')
    axes[idx].plot([true_vals.min(), true_vals.max()], [true_vals.min(), true_vals.max()], 'r--', lw=2, label="Ideal (y=x)")
    axes[idx].set_title(f"{coord} Coordinate Correlation (GBDT)")
    axes[idx].set_xlabel(f"Actual {coord} (mm)")
    axes[idx].set_ylabel(f"Predicted {coord} (mm)")
    axes[idx].grid(True, linestyle=':', alpha=0.6)
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 7. Calculate Reconstruction Efficiencies (Accuracy)
thresholds = [300.0, 500.0, 1000.0, 1500.0]
print("Vertex Reconstruction Efficiency (Accuracy) on Test Set:")
print("Threshold  | Before ML  | After ML   | Absolute Improvement")
print("-----------------------------------------------------------")
for th in thresholds:
    acc_before = np.mean(dr_before <= th) * 100.0
    acc_after = np.mean(dr_after <= th) * 100.0
    print(f"dr < {int(th):4d}mm | {acc_before:8.1f}% | {acc_after:8.1f}% | +{acc_after-acc_before:5.1f}%")